# Solution 03 — Vendor Email → PO Draft

Reference implementation for Assessment 03.


## Architectural Decisions

Pipeline: **filter/route → extract → reconcile corrections → arithmetic validate → review flags**

1. Classify which emails are order-related.
2. Extract candidate line items and header fields from order emails only.
3. Reconcile corrections ("ignore earlier…") into a final line list.
4. Deterministic Python math for totals + LLM pass for remaining human-review flags.

Extraction problems benefit from **LLM for language, code for numbers** — different from pure synthesis tasks.


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
VENDOR_EMAILS = [
  {
    "email_id": "E-1001",
    "from": "sales@northwind-components.com",
    "subject": "Re: quote follow-up / your PO?",
    "body": "Hi team,\n\nThanks for the call. As discussed please ship:\n- 40x NW-44A mounting brackets @ $12.50 each (same as last quarter)\n- 15x NW-90 cable kits \u2014 wait, making that 18 if you can, @ $27 each\n- 2x rush replacement for NW-12 seals, warranty, $0\n\nDeliver to Warehouse B, 880 Industrial Pkwy, Dock 3 by Friday next week if possible.\nBill to Acme Robotics, Net 30, attention: AP.\n\nAlso ignore my earlier email that said 25 brackets \u2014 that was wrong.\n\nThanks,\nMina (Northwind)\n"
  },
  {
    "email_id": "E-1002",
    "from": "accounts@northwind-components.com",
    "subject": "Invoice reminder \u2014 unrelated",
    "body": "Friendly reminder invoice INV-8891 for $420 is due. Please remit. This is not a new order."
  }
]


## Why Each Call

| Call | Role |
|------|------|
| Route | Drop invoice noise early |
| Extract | Pull headers + raw lines from messy prose |
| Reconcile | Resolve revisions across thread |
| Review critique | Qualitative gaps after math is fixed in code |


In [ ]:
def build_po_draft(emails: list) -> dict:
    if not emails:
        return {
            "is_purchase_order_request": False,
            "po_draft": None,
            "corrections_applied": [],
            "needs_human_review": [{"field": "emails", "reason": "empty input", "priority": "high"}],
            "ignored_emails": [],
            "validation_notes": [],
        }

    thread = json.dumps(emails, indent=2)

    # Call 1: route
    route_raw = call_llm(
        prompt=(
            "Classify each email as order-related or not. Return JSON "
            "{\"order_email_ids\":[...], \"ignored_emails\":[...], \"reason\":\"...\"}\n\n"
            + thread
        ),
        system_prompt="You triage procurement inbox threads.",
        temperature=0.0,
    )
    try:
        route = extract_json(route_raw)
    except ValueError:
        route = {
            "order_email_ids": [e["email_id"] for e in emails],
            "ignored_emails": [],
        }

    order_ids = set(route.get("order_email_ids", []))
    order_emails = [e for e in emails if e.get("email_id") in order_ids] or list(emails)
    ignored = route.get("ignored_emails", [])

    if not order_ids and route.get("ignored_emails"):
        # Likely no PO in thread
        pass

    # Call 2: extract raw fields
    extract_raw = call_llm(
        prompt=(
            "Extract purchase order header fields and line items from these emails. "
            "Include superseded quantities as separate raw mentions if present. "
            "Return JSON {vendor, bill_to, ship_to, payment_terms, requested_delivery, "
            "currency, raw_lines:[{sku, description, quantity, unit_price, charge_type, "
            "source_email_id, note}], corrections_mentioned:[string]}\n\n"
            + json.dumps(order_emails, indent=2)
        ),
        system_prompt="You extract structured procurement data. JSON only.",
        temperature=0.1,
    )
    try:
        extracted = extract_json(extract_raw)
    except ValueError:
        extracted = {}

    # Call 3: reconcile final lines
    recon_raw = call_llm(
        prompt=(
            "Reconcile raw_lines and corrections into FINAL line items after applying "
            "later corrections over earlier ones. Return JSON "
            "{\"line_items\":[{sku, description, quantity, unit_price, charge_type}], "
            "\"corrections_applied\":[...]}\n\n"
            f"EXTRACTED:\n{json.dumps(extracted, indent=2)[:8000]}"
        ),
        system_prompt="You reconcile email thread revisions for ERP entry.",
        temperature=0.0,
    )
    try:
        recon = extract_json(recon_raw)
    except ValueError:
        recon = {"line_items": extracted.get("raw_lines", []), "corrections_applied": []}

    line_items = []
    validation_notes = []
    subtotal = 0.0
    for row in recon.get("line_items", []) or []:
        try:
            qty = float(row.get("quantity") or 0)
            price = float(row.get("unit_price") or 0)
        except (TypeError, ValueError):
            qty, price = 0.0, 0.0
            validation_notes.append(f"Non-numeric qty/price for {row.get('sku')}")
        line_total = round(qty * price, 2)
        subtotal += line_total
        line_items.append(
            {
                "sku": row.get("sku"),
                "description": row.get("description"),
                "quantity": qty,
                "unit_price": price,
                "line_total": line_total,
                "charge_type": row.get("charge_type") or "standard",
            }
        )
    subtotal = round(subtotal, 2)
    validation_notes.append(f"Recomputed subtotal={subtotal}")

    po_draft = {
        "vendor": extracted.get("vendor"),
        "bill_to": extracted.get("bill_to"),
        "ship_to": extracted.get("ship_to"),
        "payment_terms": extracted.get("payment_terms"),
        "requested_delivery": extracted.get("requested_delivery"),
        "line_items": line_items,
        "currency": extracted.get("currency") or "USD",
        "subtotal": subtotal,
    }

    is_po = bool(line_items)

    # Call 4: human review flags
    review_raw = call_llm(
        prompt=(
            "Given the po_draft, list fields needing human review (missing SKU, fuzzy dates, "
            "conflicts, warranty assumptions). Return JSON "
            "{\"needs_human_review\":[{field, reason, priority}]}\n\n"
            f"PO:\n{json.dumps(po_draft, indent=2)}"
        ),
        system_prompt="You are a careful procurement reviewer.",
        temperature=0.1,
    )
    try:
        reviews = extract_json(review_raw).get("needs_human_review", [])
    except ValueError:
        reviews = []

    return {
        "is_purchase_order_request": is_po,
        "po_draft": po_draft if is_po else None,
        "corrections_applied": recon.get("corrections_applied", []),
        "needs_human_review": reviews,
        "ignored_emails": ignored,
        "validation_notes": validation_notes,
    }


## Example Execution


In [ ]:
result = build_po_draft(VENDOR_EMAILS)
print(json.dumps(result, indent=2)[:4000])


## Alternatives

- One-shot extraction: fails on corrections and arithmetic.
- Pure regex: brittle on prose.
- Full agent mailbox tools: unnecessary for provided list input.

## Tradeoffs

LLM for language, Python for money. Extra reconcile call costs latency but catches "ignore earlier" patterns.

## Common Mistakes

- Keeping qty 25 brackets.
- Dropping $0 warranty lines.
- Trusting model-added totals.
- Treating invoice reminder as a PO.

## Beyond 45 Minutes

- Vendor catalog SKU validation; ERP dry-run API; confidence per line.
